# Подведение итогов выбора связки меры сходства и модели и индексация

## Итоговая таблица лучших связок

| Модель | Конфигурация | MRR | Precision | Recall | MAP |
|--------|-------------|-----|-----------|--------|-----|
| **MiniLM** | MiniLM_euclidean | **0.7234** | **0.3480** | **0.5644** | **0.4294** |
| E5 | E5_cosine | 0.7126 | 0.3013 | 0.4922 | 0.3740 |
| USER2 | USER2_cosine | 0.7037 | 0.3200 | 0.5217 | 0.3936 |

---

## Выбор лучшей модели

MiniLM показывает лучшие результаты по всем ключевым метрикам:

1. MRR (0.7234) наивысший среди всех моделей. Поскольку в обработке запросов техподдержки важен именно первый выданный документ, выбор производился по MRR.

2. Recall (0.5644) значительно выше, чем у E5 (0.4922) и USER2 (0.5217). Модель находит больше релевантных документов в топе.

3. Precision (0.3480) лучшая точность предсказаний.

4. MAP (0.4294) лучшее среднее качество ранжирования.

Также модель значительно лучше различала тематики обращений. 

---

## Итоговое решение

Для семантического поиска для задач техподдержки выбрана итоговая связка:

- Модель: `paraphrase-multilingual-MiniLM-L12-v2`
- Мера сходства: `euclidean`
- Чанкинг: 40/20

## Выбор индексации и оценка необходимости использования реранкера

В данном ноутбуке сравнивались способы индексации: точный поиск (Flat) и графовый (HNSW). 

Сравнение производилось как на датасете только с ответами по теме, так и расширенном (240 документов) для оценки прироста скорости индексации и смысла ее использования на более обширной базе знаний, чем демонстрационная. 

---

- Flat и HNSW показали одинаковое качество поиска (MRR = 0.5912)
- Разница в скорости незначительна (0.08 ms vs 0.10 ms), поэтому был сделан вывод, что использование HNSW бессмысленно, так как такой поиск подойдет для более обширных датасетов

Сравнение индексов Flat и HNSW
| index_type | index_name | MRR | Precision | Recall | MAP | Latency (ms) |
|------------|------------|-----|-----------|--------|-----|--------------|
| flat | Flat (точный поиск) | 0.5912 | 0.2760 | 0.4483 | 0.3275 | 0.0799 |
| hnsw | HNSW (графовый) | 0.5912 | 0.2760 | 0.4483 | 0.3275 | 0.0966 |
---
Также сравнивались Flat и Flat + Reranker на чистом датасете и с добавлением лишних данных

- Реранкер не дает значительных преимуществ, однако задержка увеличивается сильно (с 0.05 ms до 454 ms)
- Точность с использованием реранкера падает, потому что он выдает много нерелевантных документов в топе

Flat vs Reranker + Flat
| dataset_type | config | reranker | MRR | Precision | Recall | MAP | Latency (ms) |
|--------------|--------|----------|-----|-----------|--------|-----|--------------|
| clean (40 docs) | Flat | False | 0.7069 | 0.3467 | 0.5628 | 0.4299 | 0.05 |
| clean (40 docs) | Flat + Reranker | True | 0.7142 | 0.1333 | 0.8683 | 0.5175 | 454.26 |
| noisy (240 docs) | Flat | False | 0.5912 | 0.2760 | 0.4483 | 0.3275 | 0.07 |
| noisy (240 docs) | Flat + Reranker | True | 0.6058 | 0.1157 | 0.7528 | 0.3975 | 459.85 |

---
**Итоги**

1. HNSW не даёт преимущества в данной ситуации. Flat оптимален по качеству и скорости
2. Реранкер нецелесообразен из-за сильной задержки и минримального увеличения MRR

Итоговая конфигурация:
- Индекс: Flat
- Реранкер: не используется
- Модель: MiniLM, мера euclidean, чанки 40/20

In [4]:
import pandas as pd
from IPython.display import display, Markdown

minilm_df = pd.read_csv('../artifacts/minilm_summary_measure_df.csv')
user2_df = pd.read_csv('../artifacts/user2_summary_measure_df.csv')
e5_df = pd.read_csv('../artifacts/e5_summary_measure_df.csv')

best_minilm = minilm_df.sort_values('mrr', ascending=False).iloc[0]
best_user2 = user2_df.sort_values('mrr', ascending=False).iloc[0]
best_e5 = e5_df.sort_values('mrr', ascending=False).iloc[0]

final_summary = pd.DataFrame([best_minilm, best_user2, best_e5])

final_summary.insert(0, 'model', ['MiniLM', 'USER2', 'E5'])

summary_cols = ['model', 'config', 'similarity', 'mrr', 'precision', 'recall', 'map']
final_summary = final_summary[summary_cols].copy()

for col in ['mrr', 'precision', 'recall', 'map']:
    final_summary[col] = final_summary[col].round(4)

final_summary = final_summary.sort_values('mrr', ascending=False)

display(Markdown(f"## Итоговая таблица лучших связок"))
display(final_summary)
final_summary.to_csv(
    "../artifacts/models_summary_bundle_df.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Файл сохранён: ./artifacts/models_summary_bundle_df.csv")

## Итоговая таблица лучших связок

,model,config,similarity,mrr,precision,recall,map
0,MiniLM,MiniLM_euclidean,euclidean,0.7234,0.3480,0.5644,0.4294
0,E5,E5_cosine,cosine,0.7126,0.3013,0.4922,0.3740
0,USER2,USER2_cosine,cosine,0.7037,0.3200,0.5217,0.3936


Файл сохранён: ./artifacts/models_summary_bundle_df.csv


In [22]:
import numpy as np
import pandas as pd
import numpy as np
import json
import pandas as pd
import faiss
from typing import List, Optional, Tuple
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display, Markdown
from sentence_transformers import CrossEncoder

In [6]:
def chunk_text(text: str, chunk_size: int = 20, overlap: int = 5) -> List[str]:
    words = text.replace("\n", " ").split()

    if chunk_size <= 0:
        raise ValueError("chunk_size должен быть положительным.")
    if overlap >= chunk_size:
        raise ValueError("overlap должен быть меньше chunk_size.")

    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(words), step):
        chunk_words = words[start : start + chunk_size]
        if not chunk_words:
            continue

        chunks.append(" ".join(chunk_words))

        if start + chunk_size >= len(words):
            break

    return chunks

def buid_chunks(
    df_docs: pd.DataFrame,
    chunk_size: Optional[int] = None,
    overlap: int = 5
) -> pd.DataFrame:

    rows = []
    
    for _, row in df_docs.iterrows():
        doc_id = row['doc_id']
        topic = row['topic']
        text = row['response_text']
        
        if chunk_size is None:
            rows.append({
                'chunk_id': doc_id,
                'doc_id': doc_id,
                'topic': topic,
                'text': text
            })
        else:
            chunks = chunk_text(text, chunk_size=chunk_size, overlap=overlap)
            for i, chunk in enumerate(chunks):
                rows.append({
                    'chunk_id': f"{doc_id}_chunk_{i}",
                    'doc_id': doc_id,
                    'topic': topic,
                    'text': chunk
                })
    
    return pd.DataFrame(rows)


def get_metrics(
    retrieved_docs: List[str],
    relevant_docs: List[str],
    metrics: List[str] = ['precision', 'recall', 'mrr', 'map']
) -> pd.DataFrame:

    relevant_set = set(relevant_docs)
    total_relevant = len(relevant_set)
    
    hits = sum(1 for doc in retrieved_docs if doc in relevant_set)
    
    result = {}
    
    if 'precision' in metrics:
        result['precision'] = hits / len(retrieved_docs) if len(retrieved_docs) > 0 else np.nan
    
    if 'recall' in metrics:
        result['recall'] = hits / total_relevant if total_relevant > 0 else np.nan
    
    if 'mrr' in metrics:
        first_relevant_rank = None
        for idx, doc_id in enumerate(retrieved_docs, start=1):
            if doc_id in relevant_docs:
                first_relevant_rank = idx
                break
        result['mrr'] = 0.0 if first_relevant_rank is None else 1.0 / first_relevant_rank
    
    if 'map' in metrics:
        precisions = []
        hits_so_far = 0
        for i, doc in enumerate(retrieved_docs, 1):
            if doc in relevant_set:
                hits_so_far += 1
                precisions.append(hits_so_far / i)
        result['map'] = sum(precisions) / total_relevant if precisions and total_relevant > 0 else 0.0
    
    return pd.DataFrame([result])

In [7]:
class EmbeddingBackend:
    def __init__(self, model_name: str, device: str = "cpu", normalize: bool = True):
        self.model = SentenceTransformer(model_name, device=device)
        self.model_name = model_name
        self.normalize = normalize
    
    def encode(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=self.normalize
        )
        return vectors.astype("float32")

def build_embedding_backend(
    model_name: str = "paraphrase-multilingual-MiniLM-L12-v2",
    device: str = "cpu",
    normalize: bool = True
) -> EmbeddingBackend:
    try:
        backend = EmbeddingBackend(model_name=model_name, device=device, normalize=normalize)
        print(f"Модель: {model_name}, нормировка={normalize}")
        return backend
    except Exception as e:
        print(f"Ошибка загрузки {model_name}: {e}")
        raise

In [31]:
class VectorSearchIndex:
    def __init__(self, dim: int, index_type: str = "flat", use_reranker: bool = False, reranker_model: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"):
        self.dim = dim
        self.index_type = index_type
        self.use_reranker = use_reranker
        self._faiss_index = None
        self._chunk_texts = None
        self._reranker = None

        if index_type == "flat":
            self._faiss_index = faiss.IndexFlatL2(dim)
        elif index_type == "hnsw":
            self._faiss_index = faiss.IndexHNSWFlat(dim, 32)
            self._faiss_index.hnsw.efSearch = 64
        if use_reranker:
            self._reranker = CrossEncoder(reranker_model, device="cpu")
            print(f"Reranker загружен: {reranker_model}")
        
    def add(self, vectors: np.ndarray, chunk_texts: List[str] = None) -> None:
        vectors = vectors.astype("float32")   
        self._faiss_index.add(vectors)
        if self.use_reranker and chunk_texts is not None:
            self._chunk_texts = chunk_texts
    
    def search(self, query_vectors: np.ndarray, query_texts: List[str] = None, top_k: int = 5, rerank_top_k: int = 20) -> Tuple[np.ndarray, np.ndarray]:
        query_vectors = query_vectors.astype("float32")
        
        search_k = rerank_top_k if self.use_reranker else top_k
        scores, indices = self._faiss_index.search(query_vectors, search_k)
        scores = -scores
        if self.use_reranker and self._chunk_texts is not None and query_texts is not None:
            reranked_scores = []
            reranked_indices = []
            
            for i, query in enumerate(query_texts):
                candidate_indices = indices[i][:rerank_top_k]
                candidate_texts = [self._chunk_texts[idx] for idx in candidate_indices]
                
                pairs = [[query, text] for text in candidate_texts]
                rerank_scores = self._reranker.predict(pairs)
                
                sorted_order = np.argsort(rerank_scores)[::-1][:top_k]
                
                reranked_indices.append([candidate_indices[idx] for idx in sorted_order])
                reranked_scores.append([rerank_scores[idx] for idx in sorted_order])
            
            return np.array(reranked_scores), np.array(reranked_indices)

        return scores, indices
        

In [36]:
import time
def evaluate_retrieval_bundle_pipeline(
    df_docs: pd.DataFrame,
    df_queries: pd.DataFrame,
    model_name: str,
    chunk_size: 40,
    overlap: int = 20,
    k: int = 5,
    index_type: str = "flat",
    use_reranker: bool = False,
    device: str = "cpu"
) -> pd.DataFrame:

    reranker_str = " + reranker" if use_reranker else ""
    print(f"Связка: модель={model_name.split('/')[-1]}, индекс={index_type}{reranker_str}")
    
    df_chunks = buid_chunks(df_docs, chunk_size=chunk_size, overlap=overlap)
    print(f"Документов/чанков: {len(df_chunks)}")

    backend = build_embedding_backend(model_name, device=device)

    chunk_texts = df_chunks['text'].tolist()
    chunk_embeddings = backend.encode(chunk_texts)
    
    dim = chunk_embeddings.shape[1]
    index = VectorSearchIndex(dim, index_type=index_type, use_reranker=use_reranker)
    
    if use_reranker:
        index.add(chunk_embeddings, chunk_texts=chunk_texts)
    else:
        index.add(chunk_embeddings)
    
    results = []
    latencies = []

    for _, row in df_queries.iterrows():
        query = row['query_text']
        relevant_docs = row['relevant_docs'].split('|')
        
        query_vec = backend.encode([query])

        start = time.time()

        if use_reranker:
            scores, indices = index.search(query_vec, query_texts=[query], top_k=k)
        else:
            scores, indices = index.search(query_vec, top_k=k)
        
        scores, indices = index.search(query_vec, top_k=k)
        latency = (time.time() - start) * 1000
        latencies.append(latency)
        
        predicted_chunk_ids = [df_chunks.iloc[idx]['chunk_id'] for idx in indices[0]]
        predicted_doc_ids = [cid.split('_chunk')[0] for cid in predicted_chunk_ids]
        
        metrics_df = get_metrics(
            retrieved_docs=predicted_doc_ids,
            relevant_docs=relevant_docs,
            metrics=['precision', 'recall', 'mrr', 'map']
        )
        
        results.append({
            'query_id': row['q_id'],
            'query': query,
            'relevant_docs': '|'.join(relevant_docs),
            'predicted_docs': '|'.join(predicted_doc_ids),
            'scores': '|'.join([f"{s:.4f}" for s in scores[0]]),
            'precision': metrics_df.iloc[0]['precision'],
            'recall': metrics_df.iloc[0]['recall'],
            'mrr': metrics_df.iloc[0]['mrr'],
            'map': metrics_df.iloc[0]['map'],
            'latency_ms': latency
        })
    
    return pd.DataFrame(results)

In [30]:
import logging
logging.getLogger("sentence_transformers").setLevel(logging.WARNING)

df_docs = pd.read_csv('../data/documents.csv')
df_docs_noise = pd.read_csv('../data/extended_documents.csv')
df_docs = pd.concat([df_docs, df_docs_noise], ignore_index=True)
df_queries = pd.read_csv('../data/queries.csv')
doc_text_map = dict(zip(df_docs['doc_id'], df_docs['response_text']))

index_configs = [
    {'name': 'Flat (точный поиск)', 'type': 'flat'},
    {'name': 'HNSW (графовый)', 'type': 'hnsw'},
]

comparison_results = []

for cfg in index_configs:
    display(Markdown(f"## Индекс: {cfg['name']}"))
    
    df_res = evaluate_retrieval_bundle_pipeline(
        df_docs=df_docs,
        df_queries=df_queries,
        model_name='paraphrase-multilingual-MiniLM-L12-v2',
        chunk_size=40,
        overlap=20,
        k=5,
        index_type=cfg['type'],
        device='cpu'
    )
    
    df_res['first_predicted_id'] = df_res['predicted_docs'].apply(lambda x: x.split('|')[0] if pd.notna(x) else '')
    df_res['first_predicted'] = df_res['first_predicted_id'].map(doc_text_map).fillna('')
    df_res['first_relevant_id'] = df_res['relevant_docs'].apply(lambda x: x.split('|')[0] if pd.notna(x) else '')
    df_res['first_hit'] = df_res['first_predicted_id'] == df_res['first_relevant_id']

    display(Markdown(f"### Результаты для индекса {cfg['name']}"))
    display_cols = ['query', 'relevant_docs', 'predicted_docs', 'scores', 'first_predicted', 'first_hit', 'mrr', 'precision', 'recall', 'map', 'latency_ms']

    display(Markdown(f"### Топ 3 самых быстрых запроса (по latency)"))
    fastest = df_res[display_cols].nsmallest(3, 'latency_ms')
    display(fastest)
    
    display(Markdown(f"### Топ 3 самых медленных запроса (по latency)"))
    slowest = df_res[display_cols].nlargest(3, 'latency_ms')
    display(slowest)
    
    avg_precision = df_res['precision'].mean()
    avg_recall = df_res['recall'].mean()
    avg_mrr = df_res['mrr'].mean()
    avg_map = df_res['map'].mean()
    avg_latency = df_res['latency_ms'].mean()
    
    comparison_results.append({
        'index_type': cfg['type'],
        'index_name': cfg['name'],
        'mrr': df_res['mrr'].mean(),
        'precision': df_res['precision'].mean(),
        'recall': df_res['recall'].mean(),
        'map': df_res['map'].mean(),
        'latency_ms': df_res['latency_ms'].mean()
    })



## Индекс: Flat (точный поиск)

Связка: модель=paraphrase-multilingual-MiniLM-L12-v2, индекс=flat
Документов/чанков: 115


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9949.89it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель: paraphrase-multilingual-MiniLM-L12-v2, нормировка=True


### Результаты для индекса Flat (точный поиск)

...

### Результаты для индекса HNSW (графовый)

### Топ 3 самых быстрых запроса (по latency)

,query,relevant_docs,predicted_docs,scores,first_predicted,first_hit,mrr,precision,recall,map,latency_ms
0,Уважаемая поддержка. Письмо для сброса пароля ...,doc_2|doc_6|doc_7,doc_1|doc_4|doc_2|doc_3|doc_7,-0.6006|-0.6403|-0.6731|-0.7194|-0.7240,Здравствуйте! Благодарим за обращение в нашу с...,False,0.333333,0.4,0.666667,0.244444,0.0
1,"Здравствуйте. Не могу войти в личный кабинет, ...",doc_1|doc_2|doc_5,doc_4|doc_3|doc_6|doc_1|doc_2,-0.3614|-0.5058|-0.5164|-0.6163|-0.6247,Приветствуем вас! Спасибо за обращение. Пробле...,False,0.250000,0.4,0.666667,0.216667,0.0
3,Привет! забыл пароль что делать подскажите,doc_1|doc_3|doc_6,doc_4|doc_6|doc_3|doc_5|doc_1,-0.4546|-0.4789|-0.4942|-0.5171|-0.5451,Приветствуем вас! Спасибо за обращение. Пробле...,False,0.500000,0.6,1.000000,0.588889,0.0


### Топ 3 самых медленных запроса (по latency)

,query,relevant_docs,predicted_docs,scores,first_predicted,first_hit,mrr,precision,recall,map,latency_ms
66,Как получить чек об оплате за прошлый месяц? В...,doc_12|doc_14|doc_16,doc_14|doc_12|doc_9|wrong_doc_23|wrong_doc_38,-0.8515|-0.9249|-0.9266|-0.9572|-0.9602,Здравствуйте! Мы получили ваше сообщение. Заде...,False,1.0,0.4,0.666667,0.666667,1.495123
2,"Не получается зайти в аккаунт, пароль не подхо...",doc_1|doc_5|doc_8,doc_4|doc_6|doc_2|doc_3|doc_5,-0.5992|-0.6869|-0.7813|-0.7948|-0.8634,Приветствуем вас! Спасибо за обращение. Пробле...,False,0.2,0.2,0.333333,0.066667,1.000881
45,Здравствуйте. Пишет ошибка 404 при переходе на...,doc_36|doc_37|doc_39,doc_36|wrong_doc_64|wrong_doc_63|doc_39|doc_37,-0.3210|-1.0875|-1.1419|-1.1710|-1.2019,Здравствуйте! Благодарим за обращение. Ошибка ...,True,1.0,0.6,1.000000,0.700000,1.000166


In [32]:
df_comparison = pd.DataFrame(comparison_results)
for col in ['mrr', 'precision', 'recall', 'map', 'latency_ms']:
    df_comparison[col] = df_comparison[col].round(4)

display(Markdown("## Сравнение индексов"))
display(df_comparison)

## Сравнение индексов

,index_type,index_name,mrr,precision,recall,map,latency_ms
0,flat,Flat (точный поиск),0.5912,0.276,0.4483,0.3275,0.0799
1,hnsw,HNSW (графовый),0.5912,0.276,0.4483,0.3275,0.0966


In [37]:
configs = [
    {'name': 'Flat', 'type': 'flat', 'reranker': False},
    {'name': 'Flat + Reranker', 'type': 'flat', 'reranker': True},
]

comparison_results = []

for cfg in configs:
    display(Markdown(f"## {cfg['name']}"))
    
    df_res = evaluate_retrieval_bundle_pipeline(
        df_docs=df_docs,
        df_queries=df_queries,
        model_name='paraphrase-multilingual-MiniLM-L12-v2',
        chunk_size=40,
        overlap=20,
        k=5,
        index_type=cfg['type'],
        use_reranker=cfg['reranker'],
        device='cpu'
    )
    
    df_res['first_predicted_id'] = df_res['predicted_docs'].apply(lambda x: x.split('|')[0] if pd.notna(x) else '')
    df_res['first_predicted'] = df_res['first_predicted_id'].map(doc_text_map).fillna('')
    df_res['first_relevant_id'] = df_res['relevant_docs'].apply(lambda x: x.split('|')[0] if pd.notna(x) else '')
    df_res['first_hit'] = df_res['first_predicted_id'] == df_res['first_relevant_id']

    display(Markdown(f"### Результаты для {cfg['name']}"))
    display_cols = ['query', 'relevant_docs', 'predicted_docs', 'scores', 'first_predicted', 'first_hit', 'mrr', 'precision', 'recall', 'map', 'latency_ms']

    display(Markdown(f"### Топ 3 самых быстрых запроса"))
    fastest = df_res[display_cols].nsmallest(3, 'latency_ms')
    display(fastest)
    
    display(Markdown(f"### Топ 3 самых медленных запроса"))
    slowest = df_res[display_cols].nlargest(3, 'latency_ms')
    display(slowest)
    
    comparison_results.append({
        'config': cfg['name'],
        'reranker': cfg['reranker'],
        'mrr': df_res['mrr'].mean(),
        'precision': df_res['precision'].mean(),
        'recall': df_res['recall'].mean(),
        'map': df_res['map'].mean(),
        'latency_ms': df_res['latency_ms'].mean()
    })


df_comparison = pd.DataFrame(comparison_results)
for col in ['mrr', 'precision', 'recall', 'map', 'latency_ms']:
    df_comparison[col] = df_comparison[col].round(4)

display(Markdown("## Сравнение Flat с и без Reranker"))
display(df_comparison)

## Flat

Связка: модель=paraphrase-multilingual-MiniLM-L12-v2, индекс=flat
Документов/чанков: 115


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9045.62it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель: paraphrase-multilingual-MiniLM-L12-v2, нормировка=True


### Результаты для Flat

...

,query,relevant_docs,predicted_docs,scores,first_predicted,first_hit,mrr,precision,recall,map,latency_ms
109,незаходит вкабинет ваще,doc_1|doc_4|doc_5,wrong_doc_28|wrong_doc_27|wrong_doc_60|wrong_d...,-1.0037|-1.0517|-1.2536|-1.2737|-1.2940|-1.315...,"Приветствуем! Спасибо, что обратились к нам. Д...",False,0.000000,0.00,0.0,0.000000,336.998701
92,Верните деньги.,doc_12|doc_13|doc_16,doc_12|wrong_doc_2|wrong_doc_33|doc_13|wrong_d...,-1.0292|-1.1297|-1.1609|-1.1654|-1.2213|-1.257...,Здравствуйте! Благодарим за обращение. Возврат...,True,1.000000,0.15,1.0,0.566667,346.004009
105,куратор малчит неделю,doc_26|doc_30|doc_31,wrong_doc_37|wrong_doc_35|wrong_doc_34|wrong_d...,-0.9674|-1.0401|-1.1234|-1.1691|-1.2053|-1.224...,Добрый день! Спасибо за вопрос. Отмена брониро...,False,0.111111,0.15,1.0,0.126706,370.998621


### Топ 3 самых медленных запроса

,query,relevant_docs,predicted_docs,scores,first_predicted,first_hit,mrr,precision,recall,map,latency_ms
7,Уважаемая техподдержка. Аккаунт заблокировали ...,doc_4|doc_5|doc_8,wrong_doc_11|doc_4|wrong_doc_18|wrong_doc_13|d...,-0.8131|-1.0193|-1.0812|-1.1183|-1.1227|-1.171...,Здравствуйте! Благодарим за обращение в службу...,False,0.5,0.15,1.000000,0.390909,588.998795
82,Уважаемая поддержка. Уведомления от платформы ...,doc_34|doc_39|doc_40,doc_39|wrong_doc_64|doc_7|wrong_doc_46|wrong_d...,-0.8285|-0.9047|-1.0916|-1.1069|-1.1295|-1.155...,Приветствуем! Спасибо за обращение. Уведомлени...,False,1.0,0.10,0.666667,0.384615,559.999228
83,"Мобильное приложение не открывает курс, вылета...",doc_35|doc_36|doc_39,doc_35|doc_38|wrong_doc_67|wrong_doc_11|wrong_...,-0.6961|-1.1193|-1.1454|-1.1627|-1.1927|-1.233...,"Приветствуем! Спасибо, что обратились к нам. М...",True,1.0,0.10,0.666667,0.384615,559.997320


## Сравнение Flat с и без Reranker

,config,reranker,mrr,precision,recall,map,latency_ms
0,Flat,False,0.5912,0.2760,0.4483,0.3275,0.0736
1,Flat + Reranker,True,0.6058,0.1157,0.7528,0.3975,459.8489


### Сравнение с использованием reranker без лишних документов

In [38]:
df_docs = pd.read_csv('../data/documents.csv')

configs = [
    {'name': 'Flat', 'type': 'flat', 'reranker': False},
    {'name': 'Flat + Reranker', 'type': 'flat', 'reranker': True},
]

comparison_results = []

for cfg in configs:
    display(Markdown(f"## {cfg['name']}"))
    
    df_res = evaluate_retrieval_bundle_pipeline(
        df_docs=df_docs,
        df_queries=df_queries,
        model_name='paraphrase-multilingual-MiniLM-L12-v2',
        chunk_size=40,
        overlap=20,
        k=5,
        index_type=cfg['type'],
        use_reranker=cfg['reranker'],
        device='cpu'
    )
    
    df_res['first_predicted_id'] = df_res['predicted_docs'].apply(lambda x: x.split('|')[0] if pd.notna(x) else '')
    df_res['first_predicted'] = df_res['first_predicted_id'].map(doc_text_map).fillna('')
    df_res['first_relevant_id'] = df_res['relevant_docs'].apply(lambda x: x.split('|')[0] if pd.notna(x) else '')
    df_res['first_hit'] = df_res['first_predicted_id'] == df_res['first_relevant_id']

    display(Markdown(f"### Результаты для {cfg['name']}"))
    display_cols = ['query', 'relevant_docs', 'predicted_docs', 'scores', 'first_predicted', 'first_hit', 'mrr', 'precision', 'recall', 'map', 'latency_ms']

    display(Markdown(f"### Топ 3 самых быстрых запроса"))
    fastest = df_res[display_cols].nsmallest(3, 'latency_ms')
    display(fastest)
    
    display(Markdown(f"### Топ 3 самых медленных запроса"))
    slowest = df_res[display_cols].nlargest(3, 'latency_ms')
    display(slowest)
    
    comparison_results.append({
        'config': cfg['name'],
        'reranker': cfg['reranker'],
        'mrr': df_res['mrr'].mean(),
        'precision': df_res['precision'].mean(),
        'recall': df_res['recall'].mean(),
        'map': df_res['map'].mean(),
        'latency_ms': df_res['latency_ms'].mean()
    })


df_comparison = pd.DataFrame(comparison_results)
for col in ['mrr', 'precision', 'recall', 'map', 'latency_ms']:
    df_comparison[col] = df_comparison[col].round(4)

display(Markdown("## Сравнение Flat с и без Reranker"))
display(df_comparison)

## Flat

Связка: модель=paraphrase-multilingual-MiniLM-L12-v2, индекс=flat
Документов/чанков: 40


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9045.72it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель: paraphrase-multilingual-MiniLM-L12-v2, нормировка=True


### Результаты для Flat

...

,query,relevant_docs,predicted_docs,scores,first_predicted,first_hit,mrr,precision,recall,map,latency_ms
91,Где сертификат?,doc_17|doc_19|doc_23,doc_23|doc_17|doc_21|doc_20|doc_19|doc_9|doc_1...,-0.6347|-0.8022|-0.8092|-0.8217|-0.9151|-1.066...,Приветствуем! Спасибо за обращение. Некоторые ...,False,1.0,0.15,1.0,0.866667,367.310524
113,почта пдтв н прхдт,doc_2|doc_6|doc_7,doc_22|doc_7|doc_10|doc_15|doc_31|doc_9|doc_11...,-1.2783|-1.3041|-1.4208|-1.4581|-1.4621|-1.483...,Здравствуйте! Мы получили ваше сообщение. Серт...,False,0.5,0.15,1.0,0.266667,370.004892
127,курс оплатил адоступа нет,doc_9|doc_12|doc_14,doc_9|doc_15|doc_12|doc_13|doc_10|doc_14|doc_1...,-0.9421|-0.9786|-0.9841|-0.9912|-1.0810|-1.184...,Здравствуйте! Спасибо за обращение. Оплата обу...,True,1.0,0.15,1.0,0.722222,374.998331


### Топ 3 самых медленных запроса

,query,relevant_docs,predicted_docs,scores,first_predicted,first_hit,mrr,precision,recall,map,latency_ms
21,Уважаемая поддержка. Сертификат с отличием выд...,doc_18|doc_21|doc_22,doc_23|doc_17|doc_21|doc_9|doc_19|doc_18|doc_2...,-0.5780|-0.7310|-0.8485|-0.8805|-0.9039|-0.919...,Приветствуем! Спасибо за обращение. Некоторые ...,False,0.333333,0.15,1.0,0.322222,552.999735
7,Уважаемая техподдержка. Аккаунт заблокировали ...,doc_4|doc_5|doc_8,doc_4|doc_5|doc_37|doc_1|doc_13|doc_8|doc_39|d...,-1.0193|-1.1227|-1.2032|-1.2132|-1.2454|-1.265...,Приветствуем вас! Спасибо за обращение. Пробле...,True,1.000000,0.15,1.0,0.833333,551.999807
30,Уважаемая поддержка. Загрузил домашнее задание...,doc_25|doc_29|doc_30,doc_25|doc_29|doc_32|doc_26|doc_27|doc_17|doc_...,-0.6649|-0.8154|-1.0580|-1.1094|-1.1320|-1.156...,Здравствуйте! Спасибо за обращение. Загрузка д...,True,1.000000,0.15,1.0,0.729167,551.999092


## Сравнение Flat с и без Reranker

,config,reranker,mrr,precision,recall,map,latency_ms
0,Flat,False,0.7069,0.3467,0.5628,0.4299,0.0533
1,Flat + Reranker,True,0.7142,0.1333,0.8683,0.5175,454.2560
